In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("ugf27.csv",dtype=str)

In [3]:
cols = list(df.columns)

In [4]:
d_col = df.iloc[:,0]
d_col

0     3.84E-05
1     4.27E-05
2     4.69E-05
3     5.12E-05
4     5.55E-05
5     5.97E-05
6     6.40E-05
7     6.83E-05
8     7.25E-05
9     7.68E-05
10    8.11E-05
11    8.53E-05
12    8.96E-05
Name: a=1.12e-05, dtype: object

In [5]:
def extract_float(header):
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", header)
    if not m:
        raise ValueError(f"No numeric value in header: {header}")
    return float(m.group())

In [6]:
if df.shape[0] < 2:
    raise ValueError("❌ CSV has too few rows")

if len(cols) % 3 != 0:
    raise ValueError("❌ Column count not divisible by 3")

num_blocks = len(cols) // 3
print(f"✅ Detected {num_blocks} experiment blocks")

✅ Detected 338 experiment blocks


In [7]:
for i in range(0, len(cols), 3):
    if not (
        cols[i].startswith("a=")
        and cols[i+1].startswith("b=")
        and cols[i+2].startswith("c=")
    ):
        raise ValueError(
            f"❌ Header order broken at columns {i}-{i+2}: "
            f"{cols[i]}, {cols[i+1]}, {cols[i+2]}"
        )

print("✅ Header pattern a,b,c verified")

✅ Header pattern a,b,c verified


In [8]:
rows = []
n_rows = df.shape[0]

In [9]:
num_blocks = len(cols) // 3

for block in range(num_blocks):
    base = block * 3

    # ---- extract parameters from headers (Option B) ----
    a_val = extract_float(cols[base])
    b_val = extract_float(cols[base + 1])
    c_val = extract_float(cols[base + 2])

    # ---- find gain column inside this block ----
    # gain column is the ODD index among (base, base+1, base+2)
    gain_idx = None
    for idx in [base, base + 1, base + 2]:
        if idx % 2 == 1:
            gain_idx = idx
            break

    if gain_idx is None:
        raise ValueError(f"No gain column found in block {block}")

    d_idx = gain_idx - 1  # even index

    # ---- extract data ----
    d_col = df.iloc[:, d_idx].astype("float64")
    gain_col = df.iloc[:, gain_idx].astype("float64")

    if len(d_col) != n_rows or len(gain_col) != n_rows:
        raise ValueError(f"Row mismatch in block {block}")

    temp = pd.DataFrame({
        "a": a_val,
        "b": b_val,
        "c": c_val,
        "d": d_col,
        "pm": gain_col
    })

    rows.append(temp)

In [10]:
pd.set_option("display.float_format", lambda x: f"{x:.15f}")


In [11]:
rows

[                   a                 b                 c                 d  \
 0  0.000011200000000 0.000132012000000 0.000002400000000 0.000038400000000   
 1  0.000011200000000 0.000132012000000 0.000002400000000 0.000042700000000   
 2  0.000011200000000 0.000132012000000 0.000002400000000 0.000046900000000   
 3  0.000011200000000 0.000132012000000 0.000002400000000 0.000051200000000   
 4  0.000011200000000 0.000132012000000 0.000002400000000 0.000055500000000   
 5  0.000011200000000 0.000132012000000 0.000002400000000 0.000059700000000   
 6  0.000011200000000 0.000132012000000 0.000002400000000 0.000064000000000   
 7  0.000011200000000 0.000132012000000 0.000002400000000 0.000068300000000   
 8  0.000011200000000 0.000132012000000 0.000002400000000 0.000072500000000   
 9  0.000011200000000 0.000132012000000 0.000002400000000 0.000076800000000   
 10 0.000011200000000 0.000132012000000 0.000002400000000 0.000081100000000   
 11 0.000011200000000 0.000132012000000 0.0000024000

In [12]:
final_df = pd.concat(rows, ignore_index=True)

In [13]:
final_df.shape

(4394, 5)

In [14]:
final_df.head(26)

,a,b,c,d,pm
0,0.000011200000000,0.000132012000000,0.000002400000000,0.000038400000000,13004823.699999999254942
1,0.000011200000000,0.000132012000000,0.000002400000000,0.000042700000000,14389267.269999999552965
2,0.000011200000000,0.000132012000000,0.000002400000000,0.000046900000000,15560052.609999999403954
3,0.000011200000000,0.000132012000000,0.000002400000000,0.000051200000000,17487860.559999998658895
4,0.000011200000000,0.000132012000000,0.000002400000000,0.000055500000000,19517781.980000000447035
5,0.000011200000000,0.000132012000000,0.000002400000000,0.000059700000000,21330874.739999998360872
6,0.000011200000000,0.000132012000000,0.000002400000000,0.000064000000000,22962364.129999998956919
7,0.000011200000000,0.000132012000000,0.000002400000000,0.000068300000000,25036822.750000000000000
8,0.000011200000000,0.000132012000000,0.000002400000000,0.000072500000000,28150012.750000000000000
9,0.000011200000000,0.000132012000000,0.000002400000000,0.000076800000000,31016649.370000001043081


In [15]:
n_d = df.shape[0]              # should be 13
n_blocks = final_df.shape[0] // n_d

print("d values:", n_d)
print("experiments:", n_blocks)


d values: 13
experiments: 338


In [16]:
final_df.groupby(["a","b","c"]).size().value_counts()


26    169
Name: count, dtype: int64

In [17]:
g = final_df.groupby(["a","b","c"]).get_group(
    tuple(final_df[["a","b","c"]].iloc[0])
)

g1 = g.iloc[:13]["pm"].values
g2 = g.iloc[13:]["pm"].values

print("Are gain sweeps identical?", (g1 == g2).all())


Are gain sweeps identical? False


In [18]:
final_df = pd.concat(rows, ignore_index=True)

print("✅ ML-ready dataset created")
print("Shape:", final_df.shape)
final_df.head(15)

✅ ML-ready dataset created
Shape: (4394, 5)


,a,b,c,d,pm
0,0.000011200000000,0.000132012000000,0.000002400000000,0.000038400000000,13004823.699999999254942
1,0.000011200000000,0.000132012000000,0.000002400000000,0.000042700000000,14389267.269999999552965
2,0.000011200000000,0.000132012000000,0.000002400000000,0.000046900000000,15560052.609999999403954
3,0.000011200000000,0.000132012000000,0.000002400000000,0.000051200000000,17487860.559999998658895
4,0.000011200000000,0.000132012000000,0.000002400000000,0.000055500000000,19517781.980000000447035
5,0.000011200000000,0.000132012000000,0.000002400000000,0.000059700000000,21330874.739999998360872
6,0.000011200000000,0.000132012000000,0.000002400000000,0.000064000000000,22962364.129999998956919
7,0.000011200000000,0.000132012000000,0.000002400000000,0.000068300000000,25036822.750000000000000
8,0.000011200000000,0.000132012000000,0.000002400000000,0.000072500000000,28150012.750000000000000
9,0.000011200000000,0.000132012000000,0.000002400000000,0.000076800000000,31016649.370000001043081


In [19]:
final_df.to_csv("pm_ml_ready.csv", index=False)


In [24]:
pm = pd.read_csv('pm_ml_ready.csv')
ugf = pd.read_csv('ugf_ml_ready.csv')
gain =pd.read_csv('gain_ml_ready.csv')


In [25]:
print(pm.shape)
print(ugf.shape)
print(gain.shape)

(4394, 5)
(4394, 5)
(4394, 5)


In [26]:
pm.head()

,a,b,c,d,pm
0,0.000011200000000,0.000132012000000,0.000002400000000,0.000038400000000,13004823.699999999254942
1,0.000011200000000,0.000132012000000,0.000002400000000,0.000042700000000,14389267.269999999552965
2,0.000011200000000,0.000132012000000,0.000002400000000,0.000046900000000,15560052.609999999403954
3,0.000011200000000,0.000132012000000,0.000002400000000,0.000051200000000,17487860.559999998658895
4,0.000011200000000,0.000132012000000,0.000002400000000,0.000055500000000,19517781.980000000447035


In [27]:
gain.head()

,a,b,c,d,gain
0,0.000011200000000,0.000132012000000,0.000002400000000,0.000038400000000,19.218120840000001
1,0.000011200000000,0.000132012000000,0.000002400000000,0.000042700000000,20.493608660000000
2,0.000011200000000,0.000132012000000,0.000002400000000,0.000046900000000,21.684998080000000
3,0.000011200000000,0.000132012000000,0.000002400000000,0.000051200000000,22.813131210000002
4,0.000011200000000,0.000132012000000,0.000002400000000,0.000055500000000,23.895182720000001


In [28]:
ugf.head()

,a,b,c,d,ugf
0,0.000011200000000,0.000132012000000,0.000002400000000,0.000038400000000,13004823.699999999254942
1,0.000011200000000,0.000132012000000,0.000002400000000,0.000042700000000,14389267.269999999552965
2,0.000011200000000,0.000132012000000,0.000002400000000,0.000046900000000,15560052.609999999403954
3,0.000011200000000,0.000132012000000,0.000002400000000,0.000051200000000,17487860.559999998658895
4,0.000011200000000,0.000132012000000,0.000002400000000,0.000055500000000,19517781.980000000447035
